# Modèle Proie-Prédateur Simplifié

Ce notebook démontre l'architecture complète de Seapopym avec un modèle à 2 espèces :
- **Espèce 1 (Proie)** : Croissance constante
- **Espèce 2 (Prédateur)** : Consomme la proie

Objectif : Observer l'évolution des biomasses sur une simulation de 30 jours.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from seapopym.blueprint import Blueprint
from seapopym.controller import SimulationController, SimulationConfig

print("✅ Imports réussis")

## 1. Définition du Modèle Scientifique

### Équations

**Proie (biomass_prey)** :
$$\frac{dP}{dt} = r - m \cdot \text{Prédateur}$$

Où :
- $r = 0.1$ : taux de croissance constant (kg/m²/jour → kg/m²/s)
- $m = 0.05$ : taux de mortalité par prédation

**Prédateur (biomass_predator)** :
$$\frac{dD}{dt} = e \cdot \text{Proie} - d$$

Où :
- $e = 0.02$ : efficacité de conversion proie → prédateur
- $d = 0.05$ : mortalité naturelle du prédateur

In [ ]:
# Fonctions de calcul des tendances

def prey_growth(biomass_prey):
    """Croissance constante de la proie."""
    # Croissance : +0.1 kg/m²/jour = 0.1/86400 kg/m²/s
    growth_rate_per_second = 0.1 / 86400
    return {"growth": biomass_prey * 0 + growth_rate_per_second}  # Constante

def predation(biomass_prey, biomass_predator):
    """Mortalité de la proie par prédation."""
    # Mortalité = -0.05 * prédateur (kg/m²/s)
    mortality_rate_per_second = -0.05 * biomass_predator / 86400
    return {"mortality": mortality_rate_per_second}

def predator_gain(biomass_prey):
    """Gain du prédateur en consommant la proie."""
    # Gain = 0.02 * proie (kg/m²/s)
    gain_rate_per_second = 0.02 * biomass_prey / 86400
    return {"gain": gain_rate_per_second}

def predator_mortality(biomass_predator):
    """Mortalité naturelle du prédateur."""
    # Mortalité = -0.05 kg/m²/jour = -0.05/86400 kg/m²/s
    mortality_rate_per_second = -0.05 / 86400
    return {"mortality": biomass_predator * 0 + mortality_rate_per_second}  # Constante

print("✅ Fonctions de calcul définies")

## 2. Configuration du Blueprint

Le Blueprint va :
1. Enregistrer les biomasses initiales comme forçages
2. Enregistrer les unités de calcul avec leurs tendances
3. Compiler le graphe de dépendances

In [ ]:
def configure_predator_prey_model(bp: Blueprint):
    """Configure le modèle proie-prédateur dans le Blueprint."""
    
    # 1. Variables d'état (forçages initiaux)
    bp.register_forcing("biomass_prey")
    bp.register_forcing("biomass_predator")
    
    # 2. Groupe Proie
    bp.register_group("Prey", [
        {
            "func": prey_growth,
            "output_mapping": {"growth": "prey_growth_rate"},
            "output_tendencies": {"growth": "biomass_prey"},  # Tendance de biomass_prey
        },
        {
            "func": predation,
            "output_mapping": {"mortality": "prey_mortality_rate"},
            "output_tendencies": {"mortality": "biomass_prey"},  # Tendance de biomass_prey
        }
    ])
    
    # 3. Groupe Prédateur
    bp.register_group("Predator", [
        {
            "func": predator_gain,
            "output_mapping": {"gain": "predator_gain_rate"},
            "output_tendencies": {"gain": "biomass_predator"},  # Tendance de biomass_predator
        },
        {
            "func": predator_mortality,
            "output_mapping": {"mortality": "predator_mortality_rate"},
            "output_tendencies": {"mortality": "biomass_predator"},  # Tendance de biomass_predator
        }
    ])

print("✅ Configuration du modèle définie")

In [ ]:
# --- Visualisation ---
plt.figure(figsize=(14, 10))

G = bp.graph
pos = nx.spring_layout(G, seed=42, k=2)

# Séparation des noeuds par type
data_nodes = [n for n in G.nodes if isinstance(n, DataNode)]
compute_nodes = [n for n in G.nodes if isinstance(n, ComputeNode)]

# Distinguer les variables initiales des produites
initial_data_nodes = [n for n in data_nodes if n.name in plan.initial_variables]
produced_data_nodes = [n for n in data_nodes if n.name in plan.produced_variables]

# Dessin des noeuds
nx.draw_networkx_nodes(G, pos, nodelist=initial_data_nodes, 
                       node_color='lightgreen', node_shape='o', 
                       node_size=800, label='Data (Initial)')
nx.draw_networkx_nodes(G, pos, nodelist=produced_data_nodes, 
                       node_color='lightblue', node_shape='o', 
                       node_size=800, label='Data (Produced)')
nx.draw_networkx_nodes(G, pos, nodelist=compute_nodes, 
                       node_color='orange', node_shape='s', 
                       node_size=1000, label='Compute')
nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=20, 
                       edge_color='gray', width=2)

# Labels
labels = {n: n.name for n in G.nodes}
nx.draw_networkx_labels(G, pos, labels, font_size=9, font_weight='bold')

plt.title("Blueprint Dependency Graph\nTuna → Shark (Inter-group dependency)", 
          fontsize=14, fontweight='bold')
plt.legend(loc='upper left', fontsize=10)
plt.axis('off')
plt.tight_layout()
plt.show()

## 3. État Initial

Grille simplifiée 1x1 avec :
- Biomasse initiale proie : 10 kg/m²
- Biomasse initiale prédateur : 2 kg/m²

In [ ]:
initial_state = xr.Dataset(
    {
        "biomass_prey": (("x"), [10.0]),
        "biomass_predator": (("x"), [2.0]),
    },
    coords={"x": [0]}
)

print("État initial :")
print(initial_state)
print("\n✅ État initial créé")

## 4. Configuration de la Simulation

Simulation de 30 jours avec un pas de temps de 1 jour.

In [ ]:
config = SimulationConfig(
    start_date=datetime(2000, 1, 1),
    end_date=datetime(2002, 1, 1),  # 30 jours
    timestep=timedelta(days=1)
)

print(f"Configuration : {config.start_date} → {config.end_date}")
print(f"Timestep : {config.timestep}")
print("\n✅ Configuration créée")

## 5. Exécution de la Simulation

In [ ]:
# Création du controller
controller = SimulationController(config)

# Setup avec notre configuration
controller.setup(configure_predator_prey_model, initial_state)

print("Setup terminé.")
print(f"Groupes fonctionnels : {list(controller.groups.keys())}")
print(f"Tendency map : {controller.execution_plan.tendency_map}")
print("\n✅ Setup réussi")

In [ ]:
controller.time_integrator.positive_vars = ["biomass_prey", "biomass_predator"]
# Stockage des résultats pour visualisation
results_prey = [initial_state["biomass_prey"].item()]
results_predator = [initial_state["biomass_predator"].item()]
days = [0]

# Run par pas de temps pour capturer l'évolution
current_day = 0
while controller._current_time < config.end_date:
    controller.step()
    controller._current_time += config.timestep
    current_day += 1
    
    # Stockage
    results_prey.append(controller.state["biomass_prey"].item())
    results_predator.append(controller.state["biomass_predator"].item())
    days.append(current_day)

print(f"\n✅ Simulation terminée : {current_day} jours")

## 6. Visualisation des Résultats

In [ ]:
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(days, results_prey, 'o-', color='green', label='Proie', linewidth=2)
plt.plot(days, results_predator, 's-', color='red', label='Prédateur', linewidth=2)
plt.xlabel('Jours', fontsize=12)
plt.ylabel('Biomasse (kg/m²)', fontsize=12)
plt.title('Évolution des Biomasses', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(results_prey, results_predator, 'o-', color='purple', linewidth=2, markersize=4)
plt.plot(results_prey[0], results_predator[0], 'go', markersize=10, label='Départ')
plt.plot(results_prey[-1], results_predator[-1], 'rs', markersize=10, label='Arrivée')
plt.xlabel('Biomasse Proie (kg/m²)', fontsize=12)
plt.ylabel('Biomasse Prédateur (kg/m²)', fontsize=12)
plt.title('Trajectoire dans l\'Espace des Phases', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Visualisation générée")

## 7. Statistiques Finales

In [ ]:
print("="*50)
print("STATISTIQUES FINALES")
print("="*50)
print(f"\nBiomasse Proie :")
print(f"  - Initiale : {results_prey[0]:.2f} kg/m²")
print(f"  - Finale   : {results_prey[-1]:.2f} kg/m²")
print(f"  - Variation : {results_prey[-1] - results_prey[0]:+.2f} kg/m²")

print(f"\nBiomasse Prédateur :")
print(f"  - Initiale : {results_predator[0]:.2f} kg/m²")
print(f"  - Finale   : {results_predator[-1]:.2f} kg/m²")
print(f"  - Variation : {results_predator[-1] - results_predator[0]:+.2f} kg/m²")

print(f"\nBiomasse Totale :")
total_init = results_prey[0] + results_predator[0]
total_final = results_prey[-1] + results_predator[-1]
print(f"  - Initiale : {total_init:.2f} kg/m²")
print(f"  - Finale   : {total_final:.2f} kg/m²")
print(f"  - Variation : {total_final - total_init:+.2f} kg/m²")
print("\n" + "="*50)

## 8. Inspection de l'Architecture

Vérifions comment le Blueprint a organisé le modèle.

In [ ]:
print("PLAN D'EXÉCUTION")
print("="*50)

for i, (group_name, tasks) in enumerate(controller.execution_plan.task_groups):
    print(f"\n{i+1}. Groupe : {group_name}")
    for task in tasks:
        print(f"   - {task.name}")
        print(f"     Entrées  : {task.input_mapping}")
        print(f"     Sorties  : {task.output_mapping}")

print("\n" + "="*50)
print("\nTENDANCES CONFIGURÉES")
print("="*50)
for var, tendencies in controller.execution_plan.tendency_map.items():
    print(f"\n{var} ← {tendencies}")

print("\n✅ Inspection terminée")